In [1]:
import ModelCatalog
from utils import model_utils
from dataset.DataLoader import create_dataloaders
from FineTuner import fine_tune
from FineTuneStrategy import FineTuneStrategy

C:\Users\sinat\anaconda3\envs\breast-cancer-agent\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\sinat\anaconda3\envs\breast-cancer-agent\lib\site-packages\torch\utils\cpp_extension.py:28: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging  # type: ignore[attr-defined]


## Default model catalog
For the default catalog I have provided the agent with 5 models. 2 of these are vision transformers and the rest are classical deep convolutional nets.

In [2]:
catalog = ModelCatalog.get_default_catalog()
print(catalog)

Models:
Model name: vit_b_16, accuracy: 81.072, params: 86.6, gflops: 17.56
Model name: swin_s, accuracy: 83.196, params: 49.6, gflops: 8.74
Model name: resnet50, accuracy: 76.13, params: 25.6, gflops: 4.09
Model name: mobilenet_v3_large, accuracy: 74.042, params: 5.5, gflops: 0.22
Model name: densenet161, accuracy: 77.138, params: 28.7, gflops: 7.73



As you can see in the above cell, the param count ranges from 5.5M to 86.6M. The goal was to provide the model with different tools, each capable of achieving differing levels of accuracy with different parameter counts.

In [3]:
models_data = [
    catalog.get_model_data("vit_b_16"),
    catalog.get_model_data("swin_s"),
    catalog.get_model_data("resnet50"),
    catalog.get_model_data("densenet161"),
    catalog.get_model_data("mobilenet_v3_large")
]
models = [model_utils.get_model(model) for model in models_data]
print(models[0])

BreastCancerModel(
  (base_model): VisionTransformer(
    (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (encoder): Encoder(
      (dropout): Dropout(p=0.0, inplace=False)
      (layers): Sequential(
        (encoder_layer_0): EncoderBlock(
          (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (self_attention): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (dropout): Dropout(p=0.0, inplace=False)
          (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): MLPBlock(
            (0): Linear(in_features=768, out_features=3072, bias=True)
            (1): GELU(approximate='none')
            (2): Dropout(p=0.0, inplace=False)
            (3): Linear(in_features=3072, out_features=768, bias=True)
            (4): Dropout(p=0.0, inplace=False)
          )
        )
        (encoder_layer_1): EncoderBlock(
          

In [4]:
train_loader, val_loader, test_loader = create_dataloaders(batch_size=8)

Created Dataframes (800, 2), (100, 2), (100, 2)


In [5]:
vit_lora = fine_tune(models[0],
                FineTuneStrategy.LORA,
                train_loader=train_loader,
                val_loader=val_loader,
                num_epochs=15,
                )
vit_lora

trainable params: 1,769,472 || all params: 87,571,973 || trainable%: 2.0206
class_weights: tensor([0.1548, 0.4595, 1.3293, 1.2407, 1.8157], device='cuda:0')

Starting fine-tuning with strategy: LORA
Device: cuda, Epochs: 15, LR: 0.0001
--------------------------------------------------------------------------------
Epoch 1/15
  Train Loss: 1.6162
  Val Loss: 1.5177
  Val Accuracy: 0.4500
  Val F1 (Weighted): 0.4523
  Val F1 (Macro): 0.2174
  Val Precision: 0.2334
  Val Recall: 0.2493
--------------------------------------------------------------------------------
Epoch 2/15
  Train Loss: 1.5190
  Val Loss: 1.5470
  Val Accuracy: 0.3300
  Val F1 (Weighted): 0.3684
  Val F1 (Macro): 0.2981
  Val Precision: 0.3855
  Val Recall: 0.3045
--------------------------------------------------------------------------------
Epoch 3/15
  Train Loss: 1.4007
  Val Loss: 1.5141
  Val Accuracy: 0.3200
  Val F1 (Weighted): 0.3440
  Val F1 (Macro): 0.2902
  Val Precision: 0.3254
  Val Recall: 0.3077
-----

{'train_losses': [1.6162267553806304,
  1.5190166270732879,
  1.4007121765613555,
  1.2803521192073821,
  1.1278307193517685,
  0.9374596765637397,
  0.7567574933171273,
  0.5940983206033706,
  0.447292927056551,
  0.34020628087222576,
  0.2788751920312643,
  0.23907673239707947,
  0.21861865099519492,
  0.19175631195306778,
  0.18342159640043973],
 'val_losses': [1.5176568214709942,
  1.5470335758649385,
  1.5140685439109802,
  1.6179737861339862,
  1.6572173421199505,
  1.6895595972354596,
  1.8443441757789025,
  2.052962165612441,
  2.0943175783524146,
  2.2777022306735697,
  2.471585677220271,
  2.4699882452304545,
  2.476992760713284,
  2.5252830179838033,
  2.5254646883561063],
 'val_accuracies': [0.45,
  0.33,
  0.32,
  0.22,
  0.24,
  0.33,
  0.32,
  0.5,
  0.44,
  0.42,
  0.53,
  0.5,
  0.5,
  0.52,
  0.52],
 'val_f1_scores': [0.45,
  0.33,
  0.32,
  0.22,
  0.24,
  0.33,
  0.32,
  0.5,
  0.44,
  0.42,
  0.53,
  0.5,
  0.5,
  0.52,
  0.52],
 'val_precision_scores': [0.23338103

In [6]:
resnet_full_ft = fine_tune(models[2],
                          strategy=FineTuneStrategy.FULL_FT,
                          train_loader=train_loader,
                          val_loader=val_loader)
resnet_full_ft

class_weights: tensor([0.1548, 0.4595, 1.3293, 1.2407, 1.8157], device='cuda:0')

Starting fine-tuning with strategy: FULL_FT
Device: cuda, Epochs: 10, LR: 0.0001
--------------------------------------------------------------------------------
Epoch 1/10
  Train Loss: 1.6028
  Val Loss: 1.5560
  Val Accuracy: 0.3100
  Val F1 (Weighted): 0.3273
  Val F1 (Macro): 0.1769
  Val Precision: 0.2075
  Val Recall: 0.2610
--------------------------------------------------------------------------------
Epoch 2/10
  Train Loss: 1.3967
  Val Loss: 1.6034
  Val Accuracy: 0.3700
  Val F1 (Weighted): 0.4151
  Val F1 (Macro): 0.2154
  Val Precision: 0.2333
  Val Recall: 0.2722
--------------------------------------------------------------------------------
Epoch 3/10
  Train Loss: 1.1214
  Val Loss: 2.1080
  Val Accuracy: 0.1100
  Val F1 (Weighted): 0.1365
  Val F1 (Macro): 0.0928
  Val Precision: 0.1370
  Val Recall: 0.1536
------------------------------------------------------------------------------

{'train_losses': [1.6027953749895096,
  1.396680069565773,
  1.1213914301991463,
  0.6415398796647787,
  0.31668913271278143,
  0.16664772409945727,
  0.11301675099879503,
  0.047125641363672914,
  0.04513598970137536,
  0.05651763382833451],
 'val_losses': [1.5559536952238817,
  1.6033730323498065,
  2.1079817093335667,
  1.9739588223970854,
  2.3833734668218174,
  2.6989641143725467,
  2.8545520351483273,
  3.310668578514686,
  2.972499207808421,
  2.845830413011404],
 'val_accuracies': [0.31,
  0.37,
  0.11,
  0.41,
  0.49,
  0.58,
  0.53,
  0.63,
  0.57,
  0.55],
 'val_f1_scores': [0.31, 0.37, 0.11, 0.41, 0.49, 0.58, 0.53, 0.63, 0.57, 0.55],
 'val_precision_scores': [0.20752941176470588,
  0.23333333333333334,
  0.1369553376906318,
  0.2622448979591837,
  0.19418103448275864,
  0.2940384615384615,
  0.22530120481927712,
  0.3811827956989247,
  0.36705882352941177,
  0.39240348692403487],
 'val_recall_scores': [0.2610119047619047,
  0.2721726190476191,
  0.15362103174603176,
  0.323

In [7]:
mobilenet_probe = fine_tune(models[4],
                            strategy=FineTuneStrategy.LINEAR_PROBE,
                            train_loader=train_loader,
                            val_loader=val_loader)
mobilenet_probe

class_weights: tensor([0.1548, 0.4595, 1.3293, 1.2407, 1.8157], device='cuda:0')

Starting fine-tuning with strategy: LINEAR_PROBE
Device: cuda, Epochs: 10, LR: 0.0001
--------------------------------------------------------------------------------
Epoch 1/10
  Train Loss: 1.6042
  Val Loss: 1.6893
  Val Accuracy: 0.2000
  Val F1 (Weighted): 0.2369
  Val F1 (Macro): 0.1365
  Val Precision: 0.1861
  Val Recall: 0.3192
--------------------------------------------------------------------------------
Epoch 2/10
  Train Loss: 1.5668
  Val Loss: 1.6983
  Val Accuracy: 0.2700
  Val F1 (Weighted): 0.3227
  Val F1 (Macro): 0.1652
  Val Precision: 0.1942
  Val Recall: 0.2814
--------------------------------------------------------------------------------
Epoch 3/10
  Train Loss: 1.5316
  Val Loss: 1.7014
  Val Accuracy: 0.2100
  Val F1 (Weighted): 0.2767
  Val F1 (Macro): 0.1441
  Val Precision: 0.1802
  Val Recall: 0.1594
-------------------------------------------------------------------------

{'train_losses': [1.6042066931724548,
  1.5668411087989806,
  1.5316148257255555,
  1.490725907087326,
  1.4860469853878022,
  1.4445349115133286,
  1.4415013813972473,
  1.4325019133090973,
  1.4250265860557556,
  1.429302569627762],
 'val_losses': [1.6892768878203173,
  1.6982845618174627,
  1.7013674424244807,
  1.7124478633587177,
  1.6822407245635986,
  1.656537257708036,
  1.6238117859913752,
  1.6006022416628325,
  1.5937040218940148,
  1.590677600640517],
 'val_accuracies': [0.2, 0.27, 0.21, 0.19, 0.2, 0.23, 0.37, 0.41, 0.43, 0.45],
 'val_f1_scores': [0.2, 0.27, 0.21, 0.19, 0.2, 0.23, 0.37, 0.41, 0.43, 0.45],
 'val_precision_scores': [0.1860640732265446,
  0.19420634920634922,
  0.18015873015873016,
  0.1706959706959707,
  0.1472972972972973,
  0.15151515151515152,
  0.1715170278637771,
  0.16416666666666666,
  0.17903225806451611,
  0.18787878787878787],
 'val_recall_scores': [0.31919642857142855,
  0.2813988095238095,
  0.159375,
  0.25,
  0.07529761904761904,
  0.09747023809